# Anytime-valid simulation analysis

This notebook loads the combined CSV outputs from `src/avi/combine_stopping_time.py` and `src/avi/combine_power.py`. The rejection cutoff is $M_t^+\ge 1/\alpha$. 

In [ ]:
from pathlib import Path
import pandas as pd
import matplotlib.pyplot as plt

ROOT = Path.cwd().parent if Path.cwd().name == 'notebooks' else Path.cwd()
AVI = ROOT / 'src' / 'avi'
stop = pd.read_csv(AVI / 'stopping_results' / 'stopping_time_summary.csv')
power = pd.read_csv(AVI / 'power_results' / 'power_summary.csv')

# Choose the displayed alpha/rho slice; change these for another panel.
PLOT_ALPHA = 0.05
PLOT_RHO = 1.0
stop_plot = stop[(stop.alpha == PLOT_ALPHA) & (stop.rho == PLOT_RHO)].copy()
power_plot = power[(power.alpha == PLOT_ALPHA) & (power.rho == PLOT_RHO)].copy()
print('stopping rows:', len(stop), 'power rows:', len(power))

## 1. Average Monte Carlo stopping time

Solid lines are the empirical $E[\min(T,N)]/N$; dashed lines are the truncated Wald comparison $\min(E[T],N)/N$. 

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for theta, g in stop_plot[stop_plot.theta > 0].groupby('theta'):
    g = g.sort_values('N')
    color = next(ax._get_lines.prop_cycler)['color']
    ax.plot(g.N, g.mean_stopping_fraction, 'o-', color=color, label=fr'$\theta={theta}$ empirical')
    ax.plot(g.N, g.wald_upper_bound_fraction, '--', color=color, label=fr'$\theta={theta}$ Wald bound')
ax.set_xscale('log'); ax.set_ylim(0, 1.05)
ax.set(xlabel='N', ylabel='stopping-time fraction', title=fr'Stopping time: $\alpha={PLOT_ALPHA}$, $\rho={PLOT_RHO}$')
ax.legend(ncol=2, fontsize=8); ax.grid(alpha=.25); plt.tight_layout()

## 2. Average Monte Carlo power

Solid lines are empirical anytime power. Dotted lines are the oracle lower bound from the terminal event $P_\theta(M_N^+>1/\alpha)$. The fixed-$N$ power is included as a reference. 

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for theta, g in power_plot[power_plot.theta > 0].groupby('theta'):
    g = g.sort_values('N')
    color = next(ax._get_lines.prop_cycler)['color']
    ax.plot(g.N, g.anytime_power, 'o-', color=color, label=fr'$\theta={theta}$ anytime')
    ax.plot(g.N, g.terminal_power_lower_bound, ':', color=color, label=fr'$\theta={theta}$ terminal lower bound')
    ax.plot(g.N, g.std_power_theoretical, '-.', color=color, alpha=.7, label=fr'$\theta={theta}$ fixed-$N$')
ax.set_xscale('log'); ax.set_ylim(0, 1.05)
ax.set(xlabel='N', ylabel='power', title=fr'Power: $\alpha={PLOT_ALPHA}$, $\rho={PLOT_RHO}$')
ax.legend(ncol=3, fontsize=8); ax.grid(alpha=.25); plt.tight_layout()

## 3. Utility difference and oracle upper bound

The solid curve is the empirical utility difference. The dashed curve is

$$\frac12\left[\frac{\min(E[T],N)}{N}-P_\theta(M_N^+>1/\alpha)-1+\mathrm{power}_{std}\right].$$

In [ ]:
fig, ax = plt.subplots(figsize=(8, 5))
for theta, g in power_plot[power_plot.theta > 0].groupby('theta'):
    g = g.sort_values('N')
    color = next(ax._get_lines.prop_cycler)['color']
    ax.plot(g.N, g.utility_difference, 'o-', color=color, label=fr'$\theta={theta}$ empirical')
    ax.plot(g.N, g.oracle_utility_difference_upper_bound, '--', color=color, label=fr'$\theta={theta}$ oracle upper bound')
ax.axhline(0, color='black', linewidth=.8)
ax.set_xscale('log')
ax.set(xlabel='N', ylabel='utility difference', title=fr'Utility difference: $\alpha={PLOT_ALPHA}$, $\rho={PLOT_RHO}$')
ax.legend(ncol=2, fontsize=8); ax.grid(alpha=.25); plt.tight_layout()